In [1]:
# Import Required Libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

import os
print("Loading Kaggle input files...")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Loading Kaggle input files...
/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
# Load Data
print("\nLoading training and test data...")
train_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")


Loading training and test data...
Training data shape: (594194, 21)
Test data shape: (254655, 20)


In [3]:
# Preprocessing Function
def preprocess_data(df, is_training=True, encoder=None, scaler=None, target_col='Churn'):
    """
    Preprocess customer churn data with consistent encoding for train/test sets.
    """
    df_processed = df.copy()
    
    # Identify feature columns (exclude id and target)
    feature_cols = [col for col in df_processed.columns if col not in ['id', target_col]]
    
    # Separate numeric and categorical features
    numeric_cols = df_processed[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_processed[feature_cols].select_dtypes(include=['object']).columns.tolist()
    
    # Handle categorical variables with one-hot encoding
    if is_training:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
        encoder = {col: df_processed[col].unique() for col in categorical_cols}
    else:
        df_dummies = pd.get_dummies(df_processed[categorical_cols], drop_first=True)
    
    # Combine numeric and encoded categorical features
    X = pd.concat([df_processed[numeric_cols].reset_index(drop=True), 
                   df_dummies.reset_index(drop=True)], axis=1)
    
    # Handle missing values and scaling
    if is_training:
        numeric_means = X[numeric_cols].mean()
        X[numeric_cols] = X[numeric_cols].fillna(numeric_means)
        scaler = StandardScaler()
        X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
    else:
        if numeric_cols:
            X[numeric_cols] = scaler.transform(X[numeric_cols])
    
    if is_training:
        y = df_processed[target_col]
        return X, y, encoder, scaler
    else:
        return X, encoder, scaler

# Apply preprocessing to training data
print("\nPreprocessing training data...")
X_train, y_train, encoder, scaler = preprocess_data(train_df, is_training=True)
print(f"Training data shape: {X_train.shape}")
print(f"Target distribution:\n{y_train.value_counts()}")

# Apply preprocessing to test data
print("\nPreprocessing test data...")
X_test, _, _ = preprocess_data(test_df, is_training=False, encoder=encoder, scaler=scaler)

# Ensure test data has same columns as training data
missing_cols = set(X_train.columns) - set(X_test.columns)
extra_cols = set(X_test.columns) - set(X_train.columns)

if missing_cols:
    for col in missing_cols:
        X_test[col] = 0

if extra_cols:
    X_test = X_test.drop(columns=extra_cols)

X_test = X_test[X_train.columns]
print(f"Final test data shape: {X_test.shape}")


Preprocessing training data...
Training data shape: (594194, 30)
Target distribution:
Churn
No     460377
Yes    133817
Name: count, dtype: int64

Preprocessing test data...
Final test data shape: (254655, 30)


In [13]:
# Feature Engineering Experiments
print("\n" + "="*70)
print("FEATURE ENGINEERING")
print("="*70)

def engineer_features(X, numeric_cols=None):
    """
    Create engineered features to enhance model performance.
    """
    X_engineered = X.copy()
    
    # Identify numeric columns if not provided
    if numeric_cols is None:
        numeric_cols = X_engineered.select_dtypes(include=[np.number]).columns.tolist()
    
    # 1. Interaction Features (for numeric columns)
    interaction_count = 0
    for i, col1 in enumerate(numeric_cols[:5]):  # Limit to top 5 for efficiency
        for col2 in numeric_cols[i+1:6]:
            if col1 in X_engineered.columns and col2 in X_engineered.columns:
                X_engineered[f'{col1}_x_{col2}'] = X_engineered[col1] * X_engineered[col2]
                interaction_count += 1
                if interaction_count >= 5:  # Limit number of interactions
                    break
        if interaction_count >= 5:
            break
    
    # 2. Polynomial Features (degree 2 for important numeric columns)
    poly_count = 0
    for col in numeric_cols[:3]:  # Top 3 numeric features
        if col in X_engineered.columns and poly_count < 3:
            X_engineered[f'{col}_squared'] = X_engineered[col] ** 2
            poly_count += 1
    
    # 3. Ratio Features (numeric interactions as ratios)
    if len(numeric_cols) >= 2:
        col1, col2 = numeric_cols[0], numeric_cols[1]
        if col1 in X_engineered.columns and col2 in X_engineered.columns:
            # Avoid division by zero
            X_engineered[f'{col1}_div_{col2}'] = np.where(
                X_engineered[col2] != 0, 
                X_engineered[col1] / (X_engineered[col2] + 1e-8),
                0
            )
    
    # 4. Statistical Features (mean, std of numeric features)
    if numeric_cols:
        X_engineered['numeric_mean'] = X_engineered[numeric_cols].mean(axis=1)
        X_engineered['numeric_std'] = X_engineered[numeric_cols].std(axis=1, skipna=True).fillna(0)
        X_engineered['numeric_max'] = X_engineered[numeric_cols].max(axis=1)
        X_engineered['numeric_min'] = X_engineered[numeric_cols].min(axis=1)
    
    return X_engineered, numeric_cols

# Apply feature engineering to training data
print(f"\nOriginal training features: {X_train.shape[1]}")
X_train_engineered, numeric_cols_list = engineer_features(X_train)
print(f"Engineered training features: {X_train_engineered.shape[1]}")
print(f"New features created: {X_train_engineered.shape[1] - X_train.shape[1]}")

# Apply same feature engineering to test data
X_test_engineered, _ = engineer_features(X_test, numeric_cols_list)

# Ensure alignment
test_missing = set(X_train_engineered.columns) - set(X_test_engineered.columns)
for col in test_missing:
    X_test_engineered[col] = 0

X_test_engineered = X_test_engineered[X_train_engineered.columns]

print(f"Final engineered test features: {X_test_engineered.shape[1]}")
print(f"All features aligned: {X_train_engineered.shape[1] == X_test_engineered.shape[1]}")
print("="*70)


FEATURE ENGINEERING

Original training features: 30
Engineered training features: 43
New features created: 13
Final engineered test features: 43
All features aligned: True


In [14]:
# Train Model with Optimized Hyperparameters
print("\nTraining model with optimized hyperparameters...")
print("="*70)
print("OPTIMIZED HYPERPARAMETERS")
print("="*70)

# Best hyperparameters found from tuning
best_params = {
    'learning_rate': 0.15,
    'max_depth': 7,
    'max_leaf_nodes': 50,
    'min_samples_leaf': 20,
    'l2_regularization': 0.1,
    'max_bins': 128,
    'random_state': 42
}

for param, value in best_params.items():
    print(f"  {param}: {value}")

# Train baseline model (without engineered features)
print("\n" + "-"*70)
print("MODEL 1: Baseline (Original Features)")
print("-"*70)
model_baseline = HistGradientBoostingClassifier(**best_params)
model_baseline.fit(X_train, y_train)
train_accuracy_baseline = model_baseline.score(X_train, y_train)
print(f"Training Accuracy: {train_accuracy_baseline:.6f}")

# Train model with engineered features
print("\n" + "-"*70)
print("MODEL 2: With Engineered Features")
print("-"*70)
model = HistGradientBoostingClassifier(**best_params)
model.fit(X_train_engineered, y_train)
train_accuracy = model.score(X_train_engineered, y_train)
print(f"Training Accuracy: {train_accuracy:.6f}")

# Compare performance
print("\n" + "-"*70)
print("PERFORMANCE COMPARISON")
print("-"*70)
accuracy_improvement = train_accuracy - train_accuracy_baseline
print(f"Baseline Accuracy: {train_accuracy_baseline:.6f}")
print(f"Engineered Accuracy: {train_accuracy:.6f}")
print(f"Improvement: {accuracy_improvement:+.6f} ({(accuracy_improvement/train_accuracy_baseline)*100:+.2f}%)")
print(f"Feature Count Increase: {X_train_engineered.shape[1]} vs {X_train.shape[1]} (+{X_train_engineered.shape[1] - X_train.shape[1]} features)")
print("="*70)


Training model with optimized hyperparameters...
OPTIMIZED HYPERPARAMETERS
  learning_rate: 0.15
  max_depth: 7
  max_leaf_nodes: 50
  min_samples_leaf: 20
  l2_regularization: 0.1
  max_bins: 128
  random_state: 42

----------------------------------------------------------------------
MODEL 1: Baseline (Original Features)
----------------------------------------------------------------------
Training Accuracy: 0.862545

----------------------------------------------------------------------
MODEL 2: With Engineered Features
----------------------------------------------------------------------
Training Accuracy: 0.863302

----------------------------------------------------------------------
PERFORMANCE COMPARISON
----------------------------------------------------------------------
Baseline Accuracy: 0.862545
Engineered Accuracy: 0.863302
Improvement: +0.000757 (+0.09%)
Feature Count Increase: 43 vs 30 (+13 features)


In [15]:
# Generate Predictions
print("\nGenerating predictions on test data...")
print("="*70)

# Get predictions from baseline model
print("Baseline Model Predictions:")
predictions_proba_baseline = model_baseline.predict_proba(X_test)
churn_probability_baseline = predictions_proba_baseline[:, 1]
print(f"  Shape: {churn_probability_baseline.shape}")
print(f"  Range: [{churn_probability_baseline.min():.6f}, {churn_probability_baseline.max():.6f}]")
print(f"  Mean: {churn_probability_baseline.mean():.6f}")

# Get predictions from engineered model
print("\nEngineered Features Model Predictions:")
predictions_proba = model.predict_proba(X_test_engineered)
churn_probability = predictions_proba[:, 1]
print(f"  Shape: {churn_probability.shape}")
print(f"  Range: [{churn_probability.min():.6f}, {churn_probability.max():.6f}]")
print(f"  Mean: {churn_probability.mean():.6f}")

# Calculate difference between predictions
prediction_diff = np.abs(churn_probability - churn_probability_baseline)
print(f"\nPrediction Differences (Engineered - Baseline):")
print(f"  Max difference: {prediction_diff.max():.6f}")
print(f"  Mean difference: {prediction_diff.mean():.6f}")
print(f"  Std difference: {prediction_diff.std():.6f}")

# Use engineered features predictions for submission
print("\nUsing engineered features model for submission.")
print("="*70)


Generating predictions on test data...
Baseline Model Predictions:
  Shape: (254655,)
  Range: [0.000298, 0.987983]
  Mean: 0.218177

Engineered Features Model Predictions:
  Shape: (254655,)
  Range: [0.000108, 0.987889]
  Mean: 0.218187

Prediction Differences (Engineered - Baseline):
  Max difference: 0.606218
  Mean difference: 0.012202
  Std difference: 0.018836

Using engineered features model for submission.


In [19]:
# Create and Save Submission File
print("\n" + "="*70)
print("CREATING SUBMISSION FILE")
print("="*70)

# Create submission DataFrame
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'Churn': churn_probability
})

print(f"Submission DataFrame created with shape: {submission_df.shape}")

# Kaggle output directory
import os
output_path = '/kaggle/working/submission.csv'

# Save submission file
try:
    submission_df.to_csv(output_path, index=False)
    print(f"\n✓ File saved to: {output_path}")
    if os.path.exists(output_path):
        file_size = os.path.getsize(output_path)
        print(f"  File size = {file_size / 1024:.2f} KB")
    else:
        print("  ERROR: File not found after save!")
except Exception as e:
    print(f"ERROR saving file: {e}")
    # Fallback to /tmp
    output_path = '/tmp/submission.csv'
    submission_df.to_csv(output_path, index=False)
    print(f"  Saved to fallback location: {output_path}")

print(f"  Number of predictions: {len(submission_df)}")

# Display preview
print(f"\nSubmission Preview (first 15 rows):")
print(submission_df.head(15).to_string())

print("\n" + "="*70)
print("SUBMISSION READY!")
print("="*70)


CREATING SUBMISSION FILE
Submission DataFrame created with shape: (254655, 2)

✓ File saved to: /kaggle/working/submission.csv
  File size = 6760.87 KB
  Number of predictions: 254655

Submission Preview (first 15 rows):
        id     Churn
0   594194  0.053394
1   594195  0.000294
2   594196  0.113728
3   594197  0.003874
4   594198  0.467166
5   594199  0.190242
6   594200  0.887885
7   594201  0.003564
8   594202  0.024633
9   594203  0.345633
10  594204  0.005562
11  594205  0.026995
12  594206  0.126281
13  594207  0.815522
14  594208  0.002865

SUBMISSION READY!
